<a href="https://colab.research.google.com/github/Yahimeen/IA_AplicadaLLama_Challenge_02_02/blob/main/Hands_On_Dise%C3%B1o_de_Agentes_Conversacionales_Challenge_02_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**


#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [2]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [3]:
# Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (596 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/loca

In [4]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [5]:
# Instalar dependencias

!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 7.5 MB/s eta 0:00:00
  

In [6]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [7]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()

In [8]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que guarda, por cada número de teléfono, el historial reciente de mensajes — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación. Esta es una implementación manual y simplificada del concepto; la lectura del Tema menciona **Llama Stack** como el framework que estandariza esta pieza (junto con herramientas y seguridad) para no reconstruirla en cada proyecto. **Nota:** el patrón exacto de integración de Llama Stack debe confirmarse contra su documentación vigente antes de grabar o dar la sesión en vivo.

In [9]:
# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

Se define la lógica que arma cada respuesta: si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento (una decisión que conviene fijar por regla, no dejarla al modelo); en cualquier otro caso, se reenvía todo el historial reciente a Llama, para que pueda resolver referencias a mensajes anteriores.

In [10]:
# Definir la funcion de generacion de respuesta con memoria

FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "agente humano"]

SYSTEM_PROMPT = (
    "Eres el asistente de agendado de citas de una clinica dental. "
    "Responde de forma breve (2-3 lineas)."
)

def generar_respuesta(numero, texto_usuario):
    texto_low = texto_usuario.lower()
    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un miembro del equipo te contacta directamente."

    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )
    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [11]:
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}

In [12]:
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}

In [19]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v26.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [14]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama

errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)
        except Exception as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "error": repr(e)})

In [15]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [21]:
# Instalar e iniciar tunel publico con ngrok
!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading
import time

# Cerrar túneles viejos de esta sesión, si existen
try:
    ngrok.kill()
except Exception:
    pass

# Si el servidor se ejecutó antes en este runtime, pedirle que termine
try:
    servidor.should_exit = True
    time.sleep(1)
except Exception:
    pass

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info",
)
servidor = uvicorn.Server(config)

def correr_servidor():
    servidor.run()

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

# Dar tiempo a Uvicorn para abrir el puerto
time.sleep(2)

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000, "http")

CALLBACK_URL = tunel_publico.public_url + "/webhook"

print("Servidor iniciado.")
print("\nConfigura en Meta:")
print("Callback URL:", CALLBACK_URL)
print("Verify Token: usa el valor del secreto WHATSAPP_VERIFY_TOKEN")
print("\nDespués suscríbete al campo 'messages' del webhook.")


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4841]
INFO:     Started server process [4841]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Servidor iniciado.

Configura en Meta:
Callback URL: https://spotter-isolating-victory.ngrok-free.dev/webhook
Verify Token: usa el valor del secreto WHATSAPP_VERIFY_TOKEN

Después suscríbete al campo 'messages' del webhook.


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje mencionando el servicio dental, y un segundo mensaje que depende del primero — para confirmar que el historial sí se está reenviando.

In [22]:
# Mandar el primer mensaje, mencionando el servicio

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "12033002267", "text": {"body": "Quiero una cita dental"}} # 5215500000000
    ]}}]}]
}

r1 = requests.post("http://0.0.0.0:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)

INFO:     127.0.0.1:44914 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: 200 {"status":"ok"}


In [ ]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215626131742", "text": {"body": "¿Tienen horario el sábado?"}}
    ]}}]}]
}

r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)
print("Mensaje 2:", r2.status_code, r2.text)

# la respuesta demora aprox. de 1 a 2 min. al ejecutar local en colab

INFO:     127.0.0.1:41048 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: 200 {"status":"ok"}


In [ ]:
# Verificar respuestas

respuestas_enviadas

[{'numero': '525626131742',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"525626131742","wa_id":"5215626131742"}],"messages":[{"id":"wamid.HBgNNTIxNTYyNjEzMTc0MhUCABEYEkY0MzQyOTczNTU3NUNBQjU5RQA="}]}'},
 {'numero': '525626131742',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"525626131742","wa_id":"5215626131742"}],"messages":[{"id":"wamid.HBgNNTIxNTYyNjEzMTc0MhUCABEYEjVBQ0I1RUJBNkEwQjE3NUJDOQA="}]}'},
 {'numero': '525626131742',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"525626131742","wa_id":"5215626131742"}],"messages":[{"id":"wamid.HBgNNTIxNTYyNjEzMTc0MhUCABEYEjhEQkFEOERGQkIxRUE3MTJGNwA="}]}'},
 {'numero': '525626131742',
  'status': 200,
  'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"525626131742","wa_id":"5215626131742"}],"messages":[{"id":"wamid.HBgNNTIxNTYyNjEzMTc0MhUCABEYEkRFRjJBM0M3RUI0OUMzQ0Q4NQA="}]}'},
 {'numero': '525626131742',
  'status': 

In [ ]:
# Ver mensajes recibidos

mensajes_recibidos

[{'object': 'whatsapp_business_account',
  'entry': [{'id': '2178708909373006',
    'changes': [{'value': {'messaging_product': 'whatsapp',
       'metadata': {'display_phone_number': '15552050818',
        'phone_number_id': '1371581399361946'},
       'contacts': [{'wa_id': '5215626131742',
         'user_id': 'MX.1622859659568820'}],
       'statuses': [{'id': 'wamid.HBgNNTIxNTYyNjEzMTc0MhUCABEYEkVCMjZFMDE1RjA4RTNGQTU3OQA=',
         'status': 'sent',
         'timestamp': '1790041093',
         'recipient_id': '5215626131742',
         'recipient_user_id': 'MX.1622859659568820',
         'pricing': {'billable': False,
          'pricing_model': 'PMP',
          'category': 'utility',
          'type': 'free_customer_service'}}]},
      'field': 'messages'}]}]},
 {'entry': [{'changes': [{'value': {'messages': [{'from': '5215626131742',
         'text': {'body': 'Quiero una cita dental'}}]}}]}]},
 {'object': 'whatsapp_business_account',
  'entry': [{'id': '2178708909373006',
    'cha

In [ ]:
# Verificar almacenamiento en Llama Stack

numero = normalizar_numero_mx("525626131742")  # tu numero real de WhatsApp
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(conversation_id=conv_id)
for item in items.data:
    print(item.role, "->", item.content)

assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='De nada, cualquier cosa. Tu cita está reservada para el sábado a las 17:30. ¡Hasta luego!', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMessageOutputContentListOpenAIResponseInputMessageContentTextOpenAIResponseInputMessageContentImageOpenAIResponseInputMessageContentFileOpenAIResponseInputMessageContentText(text='Si, muchas gracias.', type='input_text')]
assistant -> [OpenAIResponseMessageOutputContentListOpenAIResponseOutputMessageContentOutputTextOutputOpenAIResponseContentPartRefusalOpenAIResponseOutputMessageContentOutputTextOutput(text='Tenemos citas disponibles el sábado a las 16:00 y a las 17:30 horas. ¿Te parece bien el horario que te acabo de mencionar?', annotations=[], logprobs=[], type='output_text')]
user -> [OpenAIResponseMessageOutputContentListOpenAIRes

In [ ]:
# Finalizar proceso de tunelización

ngrok.kill()
print('Proceso de ngrok cerrado.')

Proceso de ngrok cerrado.
